In [6]:
import pandas as pd

print("Storage Telemetry Analysis Notebook Initialized")

Storage Telemetry Analysis Notebook Initialized


In [7]:
# Load database config from YAML
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")

raw_df = pd.read_sql_query("SELECT * FROM raw_device_metrics", engine)
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)

raw_df["timestamp"] = pd.to_datetime(raw_df["timestamp"], errors="coerce")
curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"], errors="coerce")

print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])
print("raw shape:", raw_df.shape)
print("curated shape:", curated_df.shape)
print("date range:", raw_df["timestamp"].min(), "->", raw_df["timestamp"].max())

raw_df.head()

Connected to PostgreSQL: localhost storage_analytics
raw shape: (90720, 18)
curated shape: (30240, 40)
date range: 2026-03-30 00:01:58.457739+00:00 -> 2026-04-07 22:09:23.052181+00:00


,device,timestamp,r_s,w_s,rmb_s,wmb_s,r_await,w_await,aqu_sz,util_pct,rrqm_s,wrqm_s,rareq_sz,wareq_sz,svctm,iowait_pct,source_file,ingest_run_id
0,dm-0,2026-03-30 00:01:58.457739+00:00,130.543091,79.184408,9.858905,7.593411,0.515383,1.512394,1.228957,18.985405,6.652675,8.419268,76.046825,87.252764,0.375259,0.000000,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
1,dm-0,2026-03-30 00:07:25.975778+00:00,140.798611,111.640416,13.975163,7.333685,0.722049,2.733421,0.337266,5.122736,6.204966,13.070514,54.499338,50.337116,0.314612,2.580240,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
2,dm-0,2026-03-30 00:12:41.635044+00:00,125.286126,66.371400,12.555741,6.673198,0.747056,2.052067,0.148839,25.935530,4.115189,7.280113,73.759372,81.468518,0.483813,2.334648,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
3,dm-0,2026-03-30 00:17:16.872031+00:00,100.356193,75.783214,8.572476,6.661890,0.920823,0.937210,0.292099,26.257811,8.018142,12.748040,71.675382,38.846390,0.434196,0.713070,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad
4,dm-0,2026-03-30 00:22:38.881089+00:00,108.479764,117.221000,7.843162,6.857079,1.402010,1.604284,0.796995,16.542202,12.173366,14.236511,86.284345,62.824981,0.349234,0.000000,data/raw/generated_iostat.csv,36445d12-a567-49cd-b36a-32fb1a6698ad


In [8]:
raw_profile_fields = [
    "r_s", "w_s", "rkB_s", "wkB_s", "await", "aqu_sz", "util",
    "rrqm_s", "wrqm_s", "rareq_sz", "wareq_sz", "svctm", "iowait_pct",
]
raw_profile_fields = [c for c in raw_profile_fields if c in raw_df.columns]
raw_df[raw_profile_fields].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T

,count,mean,std,min,50%,90%,95%,99%,max
r_s,90720.0,261.513221,334.692528,0.000000,155.387703,602.521359,834.192201,1641.222390,7194.375386
w_s,90720.0,202.997518,252.443769,0.000000,127.462669,443.519794,619.709965,1282.080062,6901.031797
aqu_sz,90720.0,0.882638,1.358805,0.006177,0.499165,1.903546,2.810914,6.047659,92.694137
rrqm_s,90720.0,14.507924,12.466155,0.000000,10.916551,30.779262,39.191305,57.641484,323.547530
wrqm_s,90720.0,13.728748,14.978691,0.000000,9.071315,30.171464,41.723224,69.395153,560.086858
rareq_sz,90720.0,65.653203,43.857072,0.993202,59.454498,129.363306,155.253147,193.967337,387.962469
wareq_sz,90720.0,47.046344,34.628991,1.000000,41.041664,97.494553,116.357921,145.834201,310.719907
svctm,90720.0,4.452302,29.457105,0.012569,0.446423,5.579984,16.395991,79.760210,3092.707600
iowait_pct,90720.0,4.076868,5.979850,0.000000,2.302304,9.711354,14.481713,30.507731,100.000000


In [9]:
curated_profile_fields = [
    "total_iops",
    "avg_latency_ms",
    "util_pct",
    "aqu_sz",
    "saturation_score",
    "read_ratio",
    "write_ratio",
    "throughput_mb_s",
    "iops_total",
    "merge_rate_total",
    "merge_efficiency",
    "await_ratio",
    "svctm_await_ratio",
    "queue_efficiency",
    "write_amplification",
    "iowait_pressure",
]
curated_profile_fields = [c for c in curated_profile_fields if c in curated_df.columns]
curated_df[curated_profile_fields].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T

,count,mean,std,min,50%,90%,95%,99%,max
total_iops,30240.0,415.638293,514.776042,10.174500,260.112106,888.057326,1289.372209,2682.576096,7935.754329
avg_latency_ms,30240.0,47.268516,2701.878062,0.037558,0.984769,16.485695,62.811403,491.088699,453008.805530
util_pct,30240.0,30.500694,24.195706,0.000000,24.929521,63.687810,84.846070,100.000000,100.000000
aqu_sz,30240.0,0.889370,1.332070,0.018838,0.499015,1.917880,2.852399,6.148596,43.141044
saturation_score,30240.0,40.075785,99.824605,0.000000,11.876345,95.762654,158.527502,431.868912,3871.579221
read_ratio,30240.0,0.538471,0.120597,0.000000,0.546717,0.685293,0.718384,0.776491,0.957890
write_ratio,30240.0,0.461529,0.120597,0.042110,0.453283,0.621177,0.680188,0.787998,1.000000
throughput_mb_s,30240.0,22.394165,19.269840,0.565585,16.189444,47.929243,64.845076,91.361567,141.169006
iops_total,30240.0,415.638293,514.776042,10.174500,260.112106,888.057326,1289.372209,2682.576096,7935.754329
merge_rate_total,30240.0,28.404172,27.054753,0.003373,20.421740,60.054065,78.734798,119.149335,600.710297


In [10]:
# Null and uniqueness checks for quality sanity
quality_profile = pd.DataFrame({
    "null_pct": (curated_df.isna().mean() * 100).round(2),
    "nunique": curated_df.nunique(),
}).sort_values("null_pct", ascending=False)
quality_profile.head(20)

,null_pct,nunique
device,0.0,5
timestamp,0.0,30240
r_s,0.0,30237
w_s,0.0,30240
rmb_s,0.0,30218
wmb_s,0.0,30239
r_await,0.0,30234
w_await,0.0,30222
aqu_sz,0.0,30240
util_pct,0.0,27674
